In [5]:
import pandas as pd

df = pd.read_csv("../data/raw/7282_1.csv")

# giảm dữ liệu để chạy nhanh
df = df.sample(5000, random_state=42)

df.head()

,address,categories,city,country,latitude,longitude,name,postalCode,province,reviews.date,reviews.dateAdded,reviews.doRecommend,reviews.id,reviews.rating,reviews.text,reviews.title,reviews.userCity,reviews.username,reviews.userProvince
11107,301 Reserve Ave,"Luxury Hotels,Restaurants,Hotels & Motels,Lodg...",Roanoke,US,37.256149,-79.947005,Cambria Hotel & Suites,24016,VA,2015-10-26T00:00:00Z,2016-10-13T13:22:03Z,NaN,NaN,5.0,This upscale Choice property was perfect for o...,Base for Roanoke visit,Tucson,Paul G,AZ
29089,355 Santana Row,"Hotels,Hotel, Bar, and Hotel Bar",San Jose,US,37.320980,-121.947900,Hotel Valencia Santana Row,95128,CA,2015-12-26T00:00:00Z,2016-11-17T15:13:00Z,NaN,NaN,3.0,I've stayed two nights for a business trip at ...,"Good location, large rooms, but old bathroom",The Netherlands,Joris L,MI
15961,2120 Claude Bailey Pkwy,"Travel & Transport,Hotels & Motels",Princeton,US,41.368016,-89.456966,Americinn Lodge Suites Princeton,61356,IL,2016-03-21T00:00:00Z,2017-01-08T18:34:34Z,NaN,NaN,1.0,From the warm greeting at check-in to the serv...,overall Very satisfied,Lewistown,Kenneth N,NaN
13555,19 Sombrero Blvd,Hotels,Marathon,US,24.711823,-81.080550,Sombrero Resort and Marina,33050,Grassy Key,2016-01-23T00:00:00Z,2016-11-05T19:13:20Z,NaN,NaN,2.0,Hotel is old. This might be section due to be ...,Avoid in current condition!,Boca Raton,H.Chris,FL
9008,2512 W Lincolnway,Hotels,Cheyenne,US,41.119150,-104.849490,Americas Best Value Inn,82001,WY,2016-07-24T00:00:00Z,2016-11-21T23:27:41Z,NaN,NaN,3.0,"Not the nicest motel in the world, but it was ...",Good value for the money,NaN,A Traveler,NaN


In [6]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"[^a-z\s]","",text)
    return text

df["clean_text"] = df["reviews.text"].apply(clean_text)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
CountVectorizer(
    max_features=50,
    stop_words="english"
)
word_matrix = vectorizer.fit_transform(df["clean_text"])

NameError: name 'vectorizer' is not defined

In [ ]:
word_df = pd.DataFrame(
    word_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

word_df = word_df.applymap(lambda x: 1 if x > 0 else 0)

word_df.head()

C:\Users\Tuan\AppData\Local\Temp\ipykernel_13112\3102388531.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  word_df = word_df.applymap(lambda x: 1 if x > 0 else 0)


,all,and,are,as,at,be,breakfast,but,clean,comfortable,...,this,to,very,was,we,were,when,with,would,you
0,0,1,1,0,0,0,0,1,0,0,...,1,1,1,1,0,0,0,0,0,0
1,0,1,0,0,1,1,0,1,1,0,...,1,0,1,1,0,0,0,1,0,0
2,0,1,1,1,1,0,1,0,1,0,...,1,1,0,1,1,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,1,1,0,1,1,1,0,0,1,0
4,0,1,1,0,0,0,1,1,1,0,...,0,0,0,1,0,1,0,1,0,0


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(
    word_df,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets.head()

c:\HocTap\DATA_MINING_PROJECT\venv\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.1234,(all)
1,0.6722,(and)
2,0.1328,(are)
3,0.0996,(as)
4,0.2482,(at)


In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

rules.sort_values("lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
161548,"(in, our, the)","(of, we)",0.0982,0.1472,0.0500,0.509165,3.459001,1.0,0.035545,1.737447,0.788311,0.255885,0.424443,0.424419
161549,"(of, we)","(in, our, the)",0.1472,0.0982,0.0500,0.339674,3.459001,1.0,0.035545,1.365689,0.833606,0.255885,0.267769,0.424419
161556,"(in, our)","(of, the, we)",0.1018,0.1422,0.0500,0.491159,3.454002,1.0,0.035524,1.685792,0.791005,0.257732,0.406807,0.421388
161541,"(of, the, we)","(in, our)",0.1422,0.1018,0.0500,0.351617,3.454002,1.0,0.035524,1.385293,0.828259,0.257732,0.278131,0.421388
7980,"(clean, staff)",(friendly),0.0992,0.1530,0.0522,0.526210,3.439279,1.0,0.037022,1.787711,0.787346,0.261000,0.440625,0.433693
7985,(friendly),"(clean, staff)",0.1530,0.0992,0.0522,0.341176,3.439279,1.0,0.037022,1.367286,0.837357,0.261000,0.268624,0.433693
65618,"(in, our)","(of, we)",0.1018,0.1472,0.0506,0.497053,3.376719,1.0,0.035615,1.695606,0.783628,0.255040,0.410240,0.420402
65623,"(of, we)","(in, our)",0.1472,0.1018,0.0506,0.343750,3.376719,1.0,0.035615,1.368686,0.825345,0.255040,0.269372,0.420402
191837,"(for, and, we)","(our, to, the)",0.1400,0.1180,0.0556,0.397143,3.365617,1.0,0.039080,1.463033,0.817300,0.274704,0.316489,0.434165
191852,"(our, to, the)","(for, and, we)",0.1180,0.1400,0.0556,0.471186,3.365617,1.0,0.039080,1.626282,0.796913,0.274704,0.385101,0.434165


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

tfidf = TfidfVectorizer(max_features=500)

X = tfidf.fit_transform(df["clean_text"])

kmeans = KMeans(n_clusters=5, random_state=42)

df["cluster"] = kmeans.fit_predict(X)

In [ ]:
terms = tfidf.get_feature_names_out()

for i in range(5):

    center = kmeans.cluster_centers_[i]

    words = [terms[ind] for ind in center.argsort()[-10:]]

    print("Cluster", i)
    print(words)

Cluster 0
['not', 'but', 'for', 'to', 'in', 'room', 'and', 'the', 'was', 'no']
Cluster 1
['de', 'in', 'for', 'good', 'and', 'was', 'to', 'the', 'hotel', 'great']
Cluster 2
['of', 'it', 'were', 'room', 'in', 'to', 'we', 'and', 'was', 'the']
Cluster 3
['you', 'hotel', 'of', 'are', 'this', 'in', 'to', 'and', 'is', 'the']
Cluster 4
['room', 'comfortable', 'friendly', 'clean', 'nice', 'staff', 'the', 'and', 'was', 'very']


In [ ]:
for i in range(5):

    print("\nCluster", i)

    print(df[df.cluster == i]["reviews.text"].iloc[0])


Cluster 0
Not the nicest motel in the world, but it was clean. Towels were pretty scratchy. Breakfast came with biscuits and gravy but no eggs. Shampoo, conditioner, soap and lotion are in wall pump dispensers... so good for the environment but really cheap product inside. It was a decent stop over on our way out west.

Cluster 1
Nice, clean rooms, quite area, and a bed that gave me a great nights sleep.

Cluster 2
From the warm greeting at check-in to the service at the breakfast, the visit was superb! We received our requested room number it was clean and quiet as usual. Plans are to return this fall. Overall, highly recommended.

Cluster 3
This upscale Choice property was perfect for our needs It is close to downtown so that the restaurants and attractions are easy to get to. The restaurant and bar are nice, the staff are friendly and the food was good.It is not a five star restaurant, but for our needs it fit the bill. The rooms are large and very... More

Cluster 4
Room was clean

### Phân tích luật kết hợp

Kết quả khai phá luật kết hợp cho thấy mối quan hệ giữa các từ khóa thường xuất hiện trong các đánh giá khách sạn.

Ví dụ các luật như:

(clean) -> (room)  
(area) -> (hotel)

cho thấy khi khách hàng nhắc đến sự sạch sẽ thì họ thường nhắc đến phòng. Điều này cho thấy **độ sạch của phòng là một yếu tố rất quan trọng ảnh hưởng đến sự hài lòng của khách hàng**.

Tương tự, các từ liên quan đến khu vực hoặc vị trí khách sạn thường xuất hiện cùng với đánh giá tổng thể về khách sạn.

Các luật này giúp xác định những yếu tố dịch vụ quan trọng mà khách hàng quan tâm khi viết đánh giá.